In [3]:
!pip3 install boto3

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 140 kB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 86 kB 7.7 MB/s eta 0:00:011
     |████████████████████████████████| 14.6 MB 764 kB/s eta 0:00:011
     |████████████████████████████████| 144 kB 12.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import boto3

session = boto3.Session()  # uses env/IAM credentials
sts = session.client("sts")

In [11]:
!python3 -m pip install --user psycopg2-binary

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import psycopg2
import boto3

password = "postgresduke"

conn = None
try:
    conn = psycopg2.connect(
        host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user='postgres',
        password=password,
        sslmode='verify-full',
    sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem'
    )
    cur = conn.cursor()
    cur.execute('SELECT version();')
    print(cur.fetchone()[0])
    cur.close()
except Exception as e:
    print(f"Database error: {e}")
    raise
finally:
    if conn:
        conn.close()

PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 12.4.0, 64-bit


In [2]:
import pathlib
import psycopg2

schema_path = pathlib.Path('/Users/kedarvaidya/Downloads/TA/Duke_Capstone/schema.sql')
sql = schema_path.read_text()

password = "postgresduke"  # TODO: move to env var
conn = None
try:
    conn = psycopg2.connect(
        host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user='postgres',
        password=password,
        sslmode='verify-full',
        sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem',
    )
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute(sql)
    print(f"Applied schema from: {schema_path}")
except Exception as e:
    print(f"Schema apply error: {e}")
    raise
finally:
    if conn:
        conn.close()

Applied schema from: /Users/kedarvaidya/Downloads/TA/Duke_Capstone/schema.sql


In [6]:
import pathlib
import psycopg2

seed_path = pathlib.Path('/Users/kedarvaidya/Downloads/TA/Duke_Capstone/cleanup.sql')
seed_sql = seed_path.read_text()

conn = None
try:
    conn = psycopg2.connect(
        host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user='postgres',
        password=password,
        sslmode='verify-full',
        sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem',
    )
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute(seed_sql)
    print(f"Applied seed from: {seed_path}")
except Exception as e:
    print(f"Seed apply error: {e}")
    raise
finally:
    if conn:
        conn.close()

Applied seed from: /Users/kedarvaidya/Downloads/TA/Duke_Capstone/cleanup.sql


In [3]:
conn = psycopg2.connect(
        host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user='postgres',
        password=password,
        sslmode='verify-full',
        sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem',
    )
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("""
        select table_schema, table_name
        from information_schema.tables
        where table_type = 'BASE TABLE'
          and table_schema not in ('pg_catalog', 'information_schema')
        order by table_schema, table_name;
    """)
    print(cur.fetchall())

[('public', 'cart_items'), ('public', 'carts'), ('public', 'domains'), ('public', 'organizations'), ('public', 'project_ratings'), ('public', 'project_skills'), ('public', 'project_tags'), ('public', 'projects'), ('public', 'skills'), ('public', 'tags'), ('public', 'users')]


In [9]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
    port=5432,
    database='postgres',
    user='postgres',
    password=password,
    sslmode='verify-full',
    sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem',
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("""
        select table_schema, table_name
        from information_schema.tables
        where table_type = 'BASE TABLE'
          and table_schema not in ('pg_catalog', 'information_schema')
        order by table_schema, table_name;
    """)
    tables = cur.fetchall()

for schema, table in tables:
    print(f"\n=== {schema}.{table} ===")
    query = f"select * from {schema}.{table};"
    df = pd.read_sql_query(query, conn)
    if df.empty:
        print("(no rows)")
    else:
        print(df.to_string(index=False))

conn.close()


=== public.cart_items ===
(no rows)

=== public.carts ===
(no rows)

=== public.domains ===
 id             name
  6            AI/ML
  7        Analytics
  8 Data Engineering
  9          Product
 10          Web Dev

=== public.organizations ===
 id               name      industry company_size
 13    Blue Ridge SaaS      Software       51-200
 14   Harbor Logistics     Logistics     201-1000
 15           QuillPay       Finance       51-200
 16 Solstice Utilities        Energy        1000+
 17       Atlas Health    Healthcare     201-1000
 18  Evergreen Capital       Finance     201-1000
 19          Northwind        Retail     201-1000
 20           CivicNow Public Sector       51-200
 21         BrightPath     Education     201-1000
 22        GreenBridge       Climate       51-200
 23     Medway Systems    Healthcare       51-200
 24         MetroPulse Public Sector     201-1000

=== public.project_ratings ===
(no rows)

=== public.project_skills ===
(no rows)

=== public.projec

/var/folders/xm/qd3sjdt122zgy4slwbcb9m6c0000gn/T/ipykernel_89502/2840500415.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/var/folders/xm/qd3sjdt122zgy4slwbcb9m6c0000gn/T/ipykernel_89502/2840500415.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/var/folders/xm/qd3sjdt122zgy4slwbcb9m6c0000gn/T/ipykernel_89502/2840500415.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/var/folders/xm/qd3sjdt122zgy4slwb

 id                              title                                                                                           description  duration_weeks  domain_id  organization_id   difficulty  modality cadence confidentiality  min_hours_per_week  max_hours_per_week  is_active                       created_at
  1  NLP for Customer Support Insights    Analyze support tickets and build a topic model to surface churn risk signals for service leaders.               8          6               13     Advanced    Remote    None            None                  10                  12       True 2026-01-29 18:09:17.380645+00:00
  2        Supply Chain Risk Scorecard Design a multi-factor scorecard that blends macroeconomic and vendor data to predict disruption risk.              10          7               14 Intermediate    Hybrid    None            None                   8                  10       True 2026-01-29 18:09:17.380645+00:00
  3 Product Growth Experimentation Lab  Run a funnel

/var/folders/xm/qd3sjdt122zgy4slwbcb9m6c0000gn/T/ipykernel_89502/2840500415.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/var/folders/xm/qd3sjdt122zgy4slwbcb9m6c0000gn/T/ipykernel_89502/2840500415.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/var/folders/xm/qd3sjdt122zgy4slwbcb9m6c0000gn/T/ipykernel_89502/2840500415.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/var/folders/xm/qd3sjdt122zgy4slwb

In [3]:
import psycopg2

drop_sql = """
drop table if exists project_ratings cascade;
drop table if exists project_tags cascade;
drop table if exists project_skills cascade;
drop table if exists projects cascade;
drop table if exists tags cascade;
drop table if exists skills cascade;
drop table if exists organizations cascade;
drop table if exists domains cascade;
drop table if exists instance_responses cascade;
drop table if exists sessions cascade;
"""

conn = psycopg2.connect(
    host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
    port=5432,
    database='postgres',
    user='postgres',
    password=password,
    sslmode='verify-full',
    sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem',
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(drop_sql)

print("Dropped legacy tables.")
conn.close()

Dropped legacy tables.


In [11]:
import psycopg2

dummy_sql = """
insert into client_intake_forms (
  org_name, org_industry, org_website, contact_name, contact_email,
  project_title, project_summary, project_description,
  minimum_deliverables, stretch_goals, long_term_impact,
  scope_clarity, publication_potential,
  required_skills, technical_domains, data_access, project_sector,
  supplementary_documents, video_links
) values
(
  'Blue Ridge SaaS', 'Software', 'https://blueridgesaas.example.com',
  'Ava Morris', 'ava@blueridgesaas.example.com',
  'NLP for Customer Support Insights',
  'Topic modeling for ticket triage and churn risk signals.',
  'Analyze support tickets, cluster topics, and surface churn risk patterns.',
  'Topic modeling dashboard + key drivers report.',
  'Real‑time alerting + LLM summarization.',
  'Improve retention by proactively addressing churn signals.',
  'High', 'High',
  '["Python","NLP","Topic Modeling"]'::jsonb,
  '["AI/ML","Analytics"]'::jsonb,
  'Sample ticket exports + anonymized logs.',
  'SaaS',
  '["s3://example/docs/tickets.pdf"]'::jsonb,
  '["https://youtu.be/demo1"]'::jsonb
)
,
(
  'Harbor Logistics', 'Logistics', 'https://harborlogistics.example.com',
  'Liam Nguyen', 'liam@harborlogistics.example.com',
  'Supply Chain Risk Scorecard',
  'Multi‑factor risk scoring for vendor disruption.',
  'Blend macro data + vendor KPIs to predict disruption risk.',
  'Risk scorecard + Tableau dashboard.',
  'Automated scenario simulations.',
  'Reduce disruptions via earlier risk visibility.',
  'Medium', 'Medium',
  '["SQL","Tableau","Data Modeling"]'::jsonb,
  '["Analytics"]'::jsonb,
  'Vendor KPIs + macroeconomic data.',
  'Logistics',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'QuillPay', 'Finance', 'https://quillpay.example.com',
  'Noah Patel', 'noah@quillpay.example.com',
  'Product Growth Experimentation Lab',
  'Lift trial‑to‑paid conversion through experiments.',
  'Run funnel analysis and prioritize A/B tests.',
  'Funnel analysis + experiment roadmap.',
  'Realtime experimentation dashboards.',
  'Sustained conversion gains.',
  'High', 'High',
  '["A/B Testing","Mixpanel","Roadmaps"]'::jsonb,
  '["Product","Analytics"]'::jsonb,
  'Product analytics exports.',
  'Fintech',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Solstice Utilities', 'Energy', 'https://solstice.example.com',
  'Priya Singh', 'priya@solstice.example.com',
  'Real‑Time Energy Forecast Pipeline',
  'Streaming demand forecasting + alerting.',
  'Build a pipeline to forecast hourly energy demand and trigger alerts.',
  'Streaming pipeline + forecast model.',
  'Edge deployment + auto‑retraining.',
  'Lower costs and stabilize demand.',
  'High', 'Medium',
  '["Kafka","Spark","AWS"]'::jsonb,
  '["Data Engineering","AI/ML"]'::jsonb,
  'IoT telemetry + historical demand.',
  'Energy',
  '[]'::jsonb,
  '["https://youtu.be/demo2"]'::jsonb
)
,
(
  'Atlas Health', 'Healthcare', 'https://atlashealth.example.com',
  'Maya Chen', 'maya@atlashealth.example.com',
  'Customer Churn Early Warning',
  'Predict churn from support logs + usage.',
  'Build churn detection signals and a monitoring dashboard.',
  'Churn model + dashboard.',
  'Real‑time risk alerts.',
  'Improve retention and reduce attrition.',
  'Medium', 'High',
  '["Python","NLP","Dashboard"]'::jsonb,
  '["AI/ML","Analytics"]'::jsonb,
  'Support logs + usage metrics.',
  'Healthcare',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'MetroPulse', 'Public Sector', 'https://metropulse.example.com',
  'Jada Coleman', 'jada@metropulse.example.com',
  'Urban Mobility Equity',
  'Identify transit deserts and equity gaps.',
  'Analyze ridership, service coverage, and equity metrics for neighborhoods.',
  'Equity map dashboard + policy brief.',
  'Interactive accessibility scoring.',
  'Inform equitable transit planning investments.',
  'Medium', 'High',
  '["GIS","Policy","Dashboards"]'::jsonb,
  '["GIS","Analytics"]'::jsonb,
  'GTFS feeds + census data.',
  'Public Sector',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'GreenBridge', 'Nonprofit', 'https://greenbridge.example.org',
  'Rafael Ortiz', 'rafael@greenbridge.example.org',
  'Climate Grant Impact Modeling',
  'Measure impact of sustainability grants.',
  'Model outcomes and quantify long-term impact of climate grants.',
  'Impact model + grant reporting dashboard.',
  'Scenario planning toolkit.',
  'Improve grant allocation decisions.',
  'Low', 'Medium',
  '["Causal Inference","R","Policy"]'::jsonb,
  '["Analytics","Policy"]'::jsonb,
  'Grant applications + impact surveys.',
  'Nonprofit',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Northwind Retail', 'Retail', 'https://northwind.example.com',
  'Sophie Kim', 'sophie@northwind.example.com',
  'Retail Demand Forecasting',
  'Forecast demand by region and category.',
  'Build time-series models and stocking recommendations.',
  'Demand forecast model + replenishment playbook.',
  'Automated SKU alerts.',
  'Reduce stockouts and overstock.',
  'Medium', 'Medium',
  '["Time Series","SQL","Visualization"]'::jsonb,
  '["Analytics","Retail"]'::jsonb,
  'POS sales + inventory history.',
  'Retail',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'CivicNow', 'GovTech', 'https://civicnow.example.com',
  'Ethan Brooks', 'ethan@civicnow.example.com',
  'Conversational FAQ Assistant',
  'Answer resident questions faster.',
  'Build a chatbot for FAQs and service requests.',
  'FAQ bot + handoff workflow.',
  'Multilingual support.',
  'Improve resident service response.',
  'Low', 'Medium',
  '["LLM","UX","Prompting"]'::jsonb,
  '["AI/ML","GovTech"]'::jsonb,
  'FAQ knowledge base.',
  'Public Sector',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Medway Systems', 'Healthcare', 'https://medway.example.com',
  'Oliver Grant', 'oliver@medway.example.com',
  'Hospital Readmission Insights',
  'Identify drivers of readmissions.',
  'Analyze claims and clinical data to flag readmission risks.',
  'Readmission model + insights report.',
  'Care pathway recommendations.',
  'Reduce readmissions and penalties.',
  'High', 'High',
  '["Data Viz","SQL","Storytelling"]'::jsonb,
  '["Healthcare","Analytics"]'::jsonb,
  'Claims data + EHR extracts.',
  'Healthcare',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Evergreen Capital', 'Finance', 'https://evergreen.example.com',
  'Grace Li', 'grace@evergreen.example.com',
  'ESG Signal Scoring',
  'Score ESG risks from public signals.',
  'Ingest news, filings, and third‑party ESG data.',
  'ESG scoring model + monitoring dashboard.',
  'Automated alerts for risk spikes.',
  'Improve ESG risk management.',
  'High', 'Medium',
  '["Machine Learning","Risk","Pipelines"]'::jsonb,
  '["Finance","AI/ML"]'::jsonb,
  'News feeds + ESG datasets.',
  'Finance',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Riverdale Foods', 'CPG', 'https://riverdalefoods.example.com',
  'Mila Patel', 'mila@riverdalefoods.example.com',
  'Supplier Quality Monitoring',
  'Detect supplier quality issues early.',
  'Track defect rates, audits, and shipment quality trends.',
  'Quality scorecards + alerts.',
  'Automated supplier risk signals.',
  'Reduce recalls and quality incidents.',
  'Medium', 'Low',
  '["Dashboards","Quality","SQL"]'::jsonb,
  '["Manufacturing","Analytics"]'::jsonb,
  'QA reports + inspection logs.',
  'CPG',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Harbor Health', 'Healthcare', 'https://harborhealth.example.com',
  'Nina Shah', 'nina@harborhealth.example.com',
  'Community Health Access Mapping',
  'Map care access gaps across neighborhoods.',
  'Combine clinic locations, transit, and outcomes data.',
  'Access map dashboard + recommendations.',
  'Outreach prioritization tool.',
  'Improve equitable access to care.',
  'Low', 'Medium',
  '["GIS","Public Health","Visualization"]'::jsonb,
  '["Healthcare","GIS"]'::jsonb,
  'Clinic locations + census data.',
  'Healthcare',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Nimbus Energy', 'Energy', 'https://nimbusenergy.example.com',
  'Arjun Rao', 'arjun@nimbusenergy.example.com',
  'Residential Solar Adoption Forecast',
  'Predict solar adoption by zip code.',
  'Use incentives, demographics, and utility data to forecast adoption.',
  'Adoption forecast model + outreach list.',
  'Optimized incentive targeting.',
  'Increase renewable adoption.',
  'Medium', 'Medium',
  '["Python","Time Series","GIS"]'::jsonb,
  '["Energy","Analytics"]'::jsonb,
  'Incentive data + demographic profiles.',
  'Energy',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Harbor Logistics AI', 'Logistics', 'https://harborai.example.com',
  'Leo Park', 'leo@harborai.example.com',
  'Warehouse Slotting Optimization',
  'Optimize warehouse picking routes.',
  'Model slotting strategies to reduce pick time and congestion.',
  'Slotting recommendations + simulation.',
  'Real‑time re‑slotting engine.',
  'Reduce fulfillment time and cost.',
  'High', 'Medium',
  '["Optimization","Python","Operations"]'::jsonb,
  '["Logistics","AI/ML"]'::jsonb,
  'Warehouse telemetry + order history.',
  'Logistics',
  '[]'::jsonb,
  '[]'::jsonb
)
,
(
  'Pinecone Bank', 'Finance', 'https://pineconebank.example.com',
  'Elena Garcia', 'elena@pineconebank.example.com',
  'Fraud Pattern Discovery',
  'Detect fraud rings and abnormal activity.',
  'Use graph analytics to identify suspicious patterns.',
  'Fraud graph model + investigator dashboard.',
  'Automated case triage.',
  'Reduce fraud losses.',
  'High', 'High',
  '["Graph Analytics","Python","ML"]'::jsonb,
  '["Finance","AI/ML"]'::jsonb,
  'Transactions + device fingerprints.',
  'Finance',
  '[]'::jsonb,
  '[]'::jsonb
)
on conflict (org_name) do nothing;
"""

conn = psycopg2.connect(
    host='duke-capstone.cnwm886wwadv.us-east-1.rds.amazonaws.com',
    port=5432,
    database='postgres',
    user='postgres',
    password=password,
    sslmode='verify-full',
    sslrootcert='/Users/kedarvaidya/Downloads/global-bundle.pem',
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(dummy_sql)

print("Inserted dummy rows into client_intake_forms.")
conn.close()

Inserted dummy rows into client_intake_forms.
